# RQ4 : Estimation on medicaid prescription spending by CA Zip code. 
# Methodology = CA statewide medicaid reimbursement form SDUD 
#               Get Medi-cal eligible patients by CA zip code from DHCS 
#               Calculate share of zip eligible population / total CA population 
#               share * statewide medicaid spending. 

# Keeping the suppressed data since they are not $0.

In [1]:
import pyspark
import os

In [7]:
sc = pyspark.SparkContext.getOrCreate()
print("Spark version:", sc.version)
print(sc._jvm.org.apache.hadoop.util.VersionInfo.getVersion())

Spark version: 4.2.0
3.5.0


In [8]:
# Load D5 SDUD 2024 dataset. (Medicaid)
d5_rdd = sc.textFile("gs://ca_start_up/raw_data/D5_SDUD_2024.csv")


In [10]:
# Display the names of columns 

d5_header = d5_rdd.first()
print(d5_header)

Py4JJavaError: An error occurred while calling o29.partitions.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.gs.GoogleHadoopFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2724)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3617)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3721)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3672)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:558)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:373)
	at org.apache.hadoop.mapred.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:276)
	at org.apache.hadoop.mapred.FileInputFormat.listStatus(FileInputFormat.java:245)
	at org.apache.hadoop.mapred.FileInputFormat.getSplits(FileInputFormat.java:334)
	at org.apache.spark.rdd.HadoopRDD.getPartitions(HadoopRDD.s

In [ ]:
# Split the data

d5_data = d5_rdd.map(lambda x: x.split(","))
d5_data.take(5)

[['Utilization Type',
  'State',
  'NDC',
  'Labeler Code',
  'Product Code',
  'Package Size',
  'Year',
  'Quarter',
  'Suppression Used',
  'Product Name',
  'Units Reimbursed',
  'Number of Prescriptions',
  'Total Amount Reimbursed',
  'Medicaid Amount Reimbursed',
  'Non Medicaid Amount Reimbursed'],
 ['FFSU',
  'AK',
  '00002143380',
  '00002',
  '1433',
  '80',
  '2024',
  '4',
  'false',
  'TRULICITY ',
  '00000000226.000',
  '000000108',
  '000000106607.76',
  '0000103963.72',
  '0000002644.04'],
 ['FFSU',
  'AK',
  '00002143480',
  '00002',
  '1434',
  '80',
  '2024',
  '4',
  'false',
  'TRULICITY ',
  '00000000216.000',
  '000000108',
  '000000102247.58',
  '0000097153.90',
  '0000005093.68'],
 ['FFSU',
  'AK',
  '00002143611',
  '00002',
  '1436',
  '11',
  '2024',
  '4',
  'false',
  'EMGALITY P',
  '00000000032.000',
  '000000031',
  '000000022193.94',
  '0000022193.94',
  '0000000000.00'],
 ['FFSU',
  'AK',
  '00002144509',
  '00002',
  '1445',
  '09',
  '2024',
  '4',

In [ ]:
# Check the index for columns to find which one to extract. 
d5_columns = d5_data.first()

for i, column in enumerate(d5_columns):
    print(i, column)

0 Utilization Type
1 State
2 NDC
3 Labeler Code
4 Product Code
5 Package Size
6 Year
7 Quarter
8 Suppression Used
9 Product Name
10 Units Reimbursed
11 Number of Prescriptions
12 Total Amount Reimbursed
13 Medicaid Amount Reimbursed
14 Non Medicaid Amount Reimbursed


In [ ]:
# Remove header, filter CA and 2024. 

d5_no_header = d5_data.filter(lambda x: x != d5_columns)
d5_ca = d5_no_header.filter(lambda x: x[1] == "CA") # state = x[1]
d5_ca_2024 = d5_ca.filter(lambda x: x[6] == "2024") # year = x[6]

d5_ca_2024.take(5)

[['FFSU',
  'CA',
  '00002021301',
  '00002',
  '0213',
  '01',
  '2024',
  '4',
  'false',
  'HUMULIN R ',
  '00000000393.800',
  '000000062',
  '000000002536.51',
  '0000002148.28',
  '0000000388.23'],
 ['MCOU',
  'CA',
  '00002021301',
  '00002',
  '0213',
  '01',
  '2024',
  '4',
  'false',
  'HUMULIN R ',
  '00000000323.500',
  '000000104',
  '000000001969.60',
  '0000001145.29',
  '0000000824.31'],
 ['MCOU',
  'CA',
  '00002120001',
  '00002',
  '1200',
  '01',
  '2024',
  '4',
  'false',
  'AMYVID    ',
  '00000000018.000',
  '000000017',
  '000000036898.80',
  '0000036898.80',
  '0000000000.00'],
 ['MCOU',
  'CA',
  '00002143380',
  '00002',
  '1433',
  '80',
  '2024',
  '4',
  'false',
  'TRULICITY ',
  '00000000464.500',
  '000000231',
  '000000232392.26',
  '0000232392.26',
  '0000000000.00'],
 ['FFSU',
  'CA',
  '00002143380',
  '00002',
  '1433',
  '80',
  '2024',
  '4',
  'false',
  'TRULICITY ',
  '00000028139.000',
  '000011883',
  '000013392116.96',
  '0013276120.29',


In [ ]:
print(d5_data.count())
print(d5_ca.count())
print(d5_ca_2024.count())

5308408


164099
164099


In [ ]:
# Check suppressed values. x[8] and separate them for non and suppressed records. 
# We will keep them for now. 

d5_suppression = d5_ca_2024.filter(lambda x: x[8] == "false")
d5_no_suppression = d5_ca_2024.filter(lambda x: x[8] == "true")

print(d5_suppression.count())
print(d5_no_suppression.count())


99703
64396


In [ ]:
# Get the reimbursement value to calculate estimated Medicaid spending. 
# x[12] total reimbursement =  x[13] medicaid reimbursement + x[14] non-medicaid reimbursement
# x[8] Suppression 

d5_medicaid_reimbursement = d5_ca_2024.map(lambda x: (x[8], x[13]))
d5_medicaid_reimbursement.take(5)

d5_suppressed_reimbursement = d5_medicaid_reimbursement.filter(lambda x: x[0] == "true")
d5_suppressed_reimbursement.take(5) # showing missing values. Inspect 

d5_suppressed_reimbursement.filter(lambda x: x[1] == "").count()

64396

In [ ]:
# Count missing values on suppressed records and transform to $0 to get summation calculation. 
# We don't delete the missing value rows since it is not zero value (missing)

d5_suppressed_missing_counts = d5_suppressed_reimbursement.filter(lambda x: x[1] == "").count()
print("Missing reimbursement:", d5_suppressed_missing_counts)

d5_medicaid_clean = d5_medicaid_reimbursement.map(
    lambda x: (x[0], float(x[1]) if x[1] != "" else 0.0)
)
d5_medicaid_clean.filter( lambda x: x[0] == "true").take(5)

# Calculate CA Medicaid Reimbursement 
# Get the medicaid reimbursement amt
d5_medicaid_amt = d5_medicaid_clean.map(lambda x: x[1])

# CA total 
d5_medicaid_ttl = d5_medicaid_amt.sum()

print("CA Medicaid Reimbursement 2024: $", d5_medicaid_ttl)

Missing reimbursement: 64396
CA Medicaid Reimbursement 2024: $ 15508116687.689999


In [ ]:
# Check D5 : total = medicaid + non-medicaid

d5_validation = d5_ca_2024.filter(
    lambda x: x[12] != "" and x[13] != "" and x[14] != ""
)

d5_check = d5_validation.map(
    lambda x: (float(x[12]), float(x[13]), float(x[14]))
)

d5_check.take(5)

[(2536.51, 2148.28, 388.23),
 (1969.6, 1145.29, 824.31),
 (36898.8, 36898.8, 0.0),
 (232392.26, 232392.26, 0.0),
 (13392116.96, 13276120.29, 115996.67)]

In [ ]:
# Come back for final validation. 
# 236 records have reimbursement differences greater than $0.01
# Those have not yet been investigated. 
# The statewdie Medicaid reimbursement calculation is preliminary. 

d5_mismatch = d5_check.filter(
    lambda x: abs(x[0] - (x[1] + x[2])) > 0.01
)
print("record checked", d5_check.count())
print("reimbursement mismatch", d5_mismatch.count())

d5_mismatch.take(5)

record checked 99703
reimbursement mismatch 236


[(41.0, 10238.58, 9751.08),
 (91.0, 44651.51, 44150.14),
 (102.0, 109260.18, 103132.33),
 (23.0, 21235.26, 16839.27),
 (517.0, 3215.42, 3215.42)]

26/11/26 17:52:46 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 975535 ms exceeds timeout 120000 ms
26/11/26 17:52:46 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/26 17:52:51 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora